In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import skfuzzy as fuzz

from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from umap import UMAP

from transformers import AutoTokenizer, AutoModel, logging as hf_logging
import torch

from metrics_utils import *
from preprocessing_utils import ensure_nltk_resources, preprocess_texts

ensure_nltk_resources()
hf_logging.set_verbosity_error()

seed = 42
np.random.seed(seed)

In [ ]:
df = pd.read_csv("../datasets/new/papers_20230217-20260217_Condensed_Matter_Superconductivity.csv", dtype=str)

In [ ]:
df.head(1)

In [ ]:
CONFIG = {
    "title_column": "title",
    "abstract_column": "abstract",

    "vectorizers_to_run": ["tfidf", "sentence-transformers/all-MiniLM-L6-v2", "allenai/scibert_scivocab_uncased"],

    "tfidf": dict(min_df=3, max_df=0.7, max_features=3000, ngram_range=(1, 3)),
    "svd_components": [5, 10, 15, 20, 25, 30, 35, 50, 75],
    "use_svd": True,

    "K_values": [3, 4, 5, 6, 7, 8, 9, 10, 12, 15],
    "algorithms": ["fcm", "kmeans", "agglomerative"],
}

In [ ]:
texts = (df[CONFIG["title_column"]] + " " + df[CONFIG["abstract_column"]]).tolist()
texts_clean = preprocess_texts(texts, use_pos=True)

In [ ]:
def tfidf_vectorize(texts):
    vectorizer = TfidfVectorizer(
        ngram_range=CONFIG["tfidf"]["ngram_range"],
        min_df=CONFIG["tfidf"]["min_df"],
        max_df=CONFIG["tfidf"]["max_df"],
        max_features=CONFIG["tfidf"]["max_features"],
        stop_words="english",
    )
    return vectorizer.fit_transform(texts)

def sentence_transformers_vectorize(model_name, texts):
    model = SentenceTransformer(model_name)
    X = model.encode(texts, show_progress_bar=True)
    X = normalize(X, norm="l2")
    return X

model_cache = {}
def bert_vectorize(texts, model_name, batch_size=32):
    if not model_name in model_cache:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)
        model_cache[model_name] = (tokenizer, model)
    else:
        tokenizer, model = model_cache[model_name]

    model.eval()
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        with torch.inference_mode():
            output = model(**encoded, return_dict=True)
        
        attention_mask = encoded["attention_mask"]
        token_embeddings = output.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        embeddings = (token_embeddings * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)
        
        all_embeddings.append(embeddings.cpu().numpy())
    
    X = np.vstack(all_embeddings)
    X = normalize(X, norm="l2")
    return X

In [ ]:
def vectorize(vec_name, texts_clean):
    if vec_name == "tfidf":
        return tfidf_vectorize(texts_clean)
    elif "sentence-transformers" in vec_name:
        return sentence_transformers_vectorize(vec_name, texts_clean)
    else:
        return bert_vectorize(texts_clean, vec_name)

In [ ]:
def build_X(X_base, n_comp=None, use_svd=False):
    """
    Returns final X that all algorithms will use.
    - If use_svd=False: returns dense array (Assumes it's already normalized).
    - Se use_svd=True: apply SVD with n_comp.
    """

    if not use_svd:
        return X_base.toarray() if hasattr(X_base, "toarray") else np.asarray(X_base)
    
    svd = TruncatedSVD(n_components=n_comp, random_state=seed)
    X = svd.fit_transform(X_base)
    X = normalize(X, norm="l2")
    return X

In [ ]:
def cluster_fcm(X, K, m=1.7, error=0.005, maxiter=1000):
    cntr, U, *_rest, fpc = fuzz.cluster.cmeans(
        X.T, c=K, m=m, error=error, maxiter=maxiter,
        metric="cosine", seed=seed
    )
    U = U.T
    labels = U.argmax(axis=1)
    return labels, {"fpc": float(fpc), "U": U}

def cluster_kmeans(X, K, max_iter=300):
    model = KMeans(n_clusters=K, random_state=seed, init="k-means++", max_iter=max_iter)
    labels = model.fit_predict(X)
    return labels, {"inertia": float(model.inertia_)}

def cluster_agglomerative_cluster(X, K):
    model = AgglomerativeClustering(n_clusters=K, metric="euclidean", linkage="ward")
    labels = model.fit_predict(X)
    return labels


In [ ]:
# For cmeans only
def diagnose_collapse(U):
    eps = 1e-12
    K = U.shape[1]
    entropy = -np.sum(U * np.log(U + eps), axis=1)
    entropy_norm = float(np.mean(entropy) / np.log(K))
    avg_max_memb = float(U.max(axis=1).mean())
    return {
        "collapsed": entropy_norm > 0.85,
        "entropy_norm": entropy_norm,
        "avg_max_memb": avg_max_memb,
    }

def evaluate_all(X, y_pred):
    return {
        "CHI": calculate_chi(X, y_pred),
        "DBI": calculate_dbi(X, y_pred),
        "SIL": calculate_silhouette(X, y_pred)
    }

In [ ]:
def append_result(X, base_row, alg_name, y_pred, extras):
    extras = extras or {}
    metrics = evaluate_all(X, y_pred)
    row = {**base_row, "alg": alg_name, **metrics, **extras}
    rows.append(row)

In [ ]:
rows = []
n_comp_list = CONFIG["svd_components"] if CONFIG["use_svd"] else [None]

for vec_name in CONFIG["vectorizers_to_run"]:
    print(f"Running vectorizer: {vec_name}")

    X_base = vectorize(vec_name, texts_clean)
    n_comp_list = CONFIG["svd_components"] if CONFIG["use_svd"] else [None]

    for n_comp in n_comp_list:
        X = build_X(X_base, n_comp=n_comp, use_svd=CONFIG["use_svd"])

        for K in CONFIG["K_values"]:
            base_row = {
                "vectorizer": vec_name,
                "use_svd": CONFIG["use_svd"],
                "n_comp": n_comp,
                "K": K,
                "collapsed": False,
                "entropy_norm": np.nan,
                "avg_max_memb": np.nan,
                "fpc": np.nan,
                "inertia": np.nan,
            }

            if "fcm" in CONFIG["algorithms"]:
                y_pred, extra = cluster_fcm(X, K)
                collapse = diagnose_collapse(extra["U"])
                append_result(
                    X, base_row,
                    "FCM", y_pred,
                    {**collapse, "fpc": extra["fpc"]}
                )

            if "kmeans" in CONFIG["algorithms"]:
                y_pred, extra = cluster_kmeans(X, K)
                append_result(
                    X, base_row,
                    "KMeans", y_pred,
                    {"inertia": extra["inertia"]}
                )

            if "agglomerative" in CONFIG["algorithms"]:
                y_pred = cluster_agglomerative_cluster(X, K)
                append_result(X, base_row, "AGG", y_pred, None)

results_all = pd.DataFrame(rows).reset_index(drop=True)

In [ ]:
filtered = results_all[~((results_all["alg"] == "FCM") & (results_all["collapsed"] == True))]

In [ ]:
top_per_alg = (
    filtered
    .sort_values(
        by=["SIL", "CHI", "DBI"], 
        ascending=[False, False, True]
    )
)

top_per_alg.head(5)

In [ ]:
best_per_group = (
    filtered
    .sort_values(
        by=["SIL", "CHI", "DBI"], 
        ascending=[False, False, True]
    )
    .groupby(["vectorizer", "alg"], as_index=False)
    .head(1)
    .sort_values(["vectorizer", "alg"]) 
    .reset_index(drop=True)
)

best_per_group

In [ ]:
global_ranking = (
    best_per_group
    .sort_values(["SIL", "CHI", "DBI"], ascending=[False, False, True])
    .reset_index(drop=True)
)


global_ranking

In [ ]:
def run_one_config(row, texts_clean):
    """Rebuild X and predicted labels (y_pred) from a single row of the dataframe"""

    vec_name = row["vectorizer"]
    alg = row["alg"]
    K = int(row["K"])

    X_base = vectorize(vec_name, texts_clean)

    n_comp = None if (row["n_comp"] is None or np.isnan(row["n_comp"])) else int(row["n_comp"])
    X = build_X(X_base, n_comp=n_comp, use_svd=row["use_svd"])

    if alg == "FCM":
        y_pred, extra = cluster_fcm(X, K)
        return X, y_pred

    if alg == "KMeans":
        y_pred, extra = cluster_kmeans(X, K)
        return X, y_pred

    if alg == "AGG":
        y_pred = cluster_agglomerative_cluster(X, K)
        return X, y_pred

def reduce_to_2d(X, K, method="umap", seed=42):
    n_neighbors  = X.shape[0] // K
    
    if method.lower() == "umap":

        reducer = UMAP(
            n_components=2,
            n_neighbors=n_neighbors,
            random_state=seed,
        )
        return reducer.fit_transform(X)

    pca = PCA(n_components=2, random_state=seed)
    return pca.fit_transform(X)

In [ ]:
from math import ceil


def plot_clusters(
    best_per_group,
    texts_clean,
    method="umap",
    seed=42,
    alpha=0.75,
    point_size=12,
    figsize_per_plot=(5, 4),
):
    n_plots = len(best_per_group)
    n_cols = 3
    n_rows = ceil(n_plots / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_per_plot[0] * n_cols, figsize_per_plot[1] * n_rows),
        squeeze=False,
    )

    axes = axes.flatten()

    for ax, (_, row) in zip(axes, best_per_group.iterrows()):
        # Rebuild X and labels for this configuration
        X, y_pred = run_one_config(row, texts_clean)
        K = int(row["K"])

        Z = reduce_to_2d(X, K, method=method, seed=seed)

        sc = ax.scatter(
            Z[:, 0],
            Z[:, 1],
            c=y_pred,
            s=point_size,
            alpha=alpha,
        )

        n_comp = row.get("n_comp", None)
        if n_comp is None or np.isnan(n_comp):
            n_comp_str = ""
        else:
            n_comp_str = f", n_comp={int(n_comp)}"

        title = (
            f"{row['vectorizer']} | {row['alg']} | K={int(row['K'])}{n_comp_str}\n"
            f"CHI={row['CHI']:.3f} DBI={row['DBI']:.3f} SIL={row['SIL']:.3f}"
        )

        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

    # Turn off unused axes (if grid is larger than number of plots)
    for ax in axes[n_plots:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_clusters(best_per_group, texts_clean)